# Composite DIMSIM geometry analysis

Imports the exact checked canonical implementation from `CODE_ROOT`.


In [ ]:
BUILD_METADATA = {'schema_version': 1, 'baseline_git_sha': '977d20f1b06119956b5ea7676654eff2df89d719', 'source_sha256': {'evaluation/dimsim.py': '38712d92358e8378539935fd669f5e9a375193010007a3798faaea02c88e0195', 'evaluation/sampling.py': '39c473e643f1f30c62951d28d4e5b6004ed819256676ff351355d098ebeab0c8', 'evaluation/bundle.py': '93646514c7f117547c21693fa8ce610d26034a7a75a259ae922462ee51339b45', 'evaluation/representations.py': '8d41e607938a592559f77a4e3900ddda3c15975017d0589fbad59a634c43ddbb', 'model.py': '9f6bd2de95cec891c875280e518726ec1c9f6f0ec6080b55fe564a88170415af', 'evaluation/encoders.py': '93444a8a740948ca06632cf68792377995fd0d88f6b152f68640e95d21a04aa4', 'evaluation/analysis.py': 'c06233981812e189e8b696c47fbe5d53d665a3d20bb611f0db15dffc12345577', 'evaluation/runner.py': 'a48b4e0478cf021a89f460641e94c559ca5f50f223e3dc7596a3a47e53d81e35'}, 'embedded_sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'generator_sha256': '4f1e588bcaf11266fe78efc1479fb90e50d652b67f58945970b6d6a0be85566d'}


In [ ]:
import hashlib
import json
import sys
from pathlib import Path

CODE_ROOT = Path("/EDIT/project")
if "EDIT" in str(CODE_ROOT) or not (CODE_ROOT / "src" / "phonetics").is_dir():
    raise ValueError("Edit CODE_ROOT to the project directory before running analysis")
if any(name == "phonetics" or name.startswith("phonetics.") for name in sys.modules):
    raise RuntimeError("Composite evaluation requires a fresh kernel before importing phonetics")
sys.path.insert(0, str(CODE_ROOT / "src"))
from phonetics.evaluation.runner import run_evaluation
from phonetics.evaluation.analysis import compare_paired_runs

for relative, expected in BUILD_METADATA["source_sha256"].items():
    actual = hashlib.sha256((CODE_ROOT / "src" / "phonetics" / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Local source differs from notebook build: {relative}")


In [ ]:
from pathlib import Path

BUNDLE_PATH = Path("/EDIT/evaluation-bundle")
OUTPUT_ROOT = Path("/EDIT/evaluation-runs")
RUNS = [
    {"run_id": "lambda-0", "adapter": "bilstm", "experiment_path": Path("/EDIT/experiment"), "checkpoint_path": Path("/EDIT/lambda-0.pt")},
    {"run_id": "lambda-infinity", "adapter": "bilstm", "experiment_path": Path("/EDIT/experiment"), "checkpoint_path": Path("/EDIT/lambda-infinity.pt")},
    {"run_id": "random-seed-42", "adapter": "random", "random_seed": 42, "random_dimension": 1024},
]
DEVICE = "cpu"
BATCH_SIZE = 32
PAIRED_COMPARISONS = [("lambda-0", "lambda-infinity")]

if "EDIT" in str(BUNDLE_PATH) or "EDIT" in str(OUTPUT_ROOT) or any("EDIT" in str(value) for run in RUNS for value in run.values()):
    raise ValueError("Edit BUNDLE_PATH, OUTPUT_ROOT, and all model input paths before running analysis")


In [ ]:
completed = {}
for specification in RUNS:
    arguments = dict(specification)
    run_id = arguments.pop("run_id")
    adapter = arguments.pop("adapter")
    completed[run_id] = run_evaluation(
        BUNDLE_PATH, output_root=OUTPUT_ROOT, run_id=run_id, adapter=adapter,
        device=DEVICE, batch_size=BATCH_SIZE, code_provenance=BUILD_METADATA["source_sha256"], **arguments,
    )

for left, right in PAIRED_COMPARISONS:
    compare_paired_runs(
        completed[left] / "analysis", completed[right] / "analysis",
        output_dir=OUTPUT_ROOT / f"compare-{left}-vs-{right}",
    )

for run_id, path in completed.items():
    metrics = json.loads((path / "analysis" / "metrics.json").read_text(encoding="utf-8"))
    print(run_id, {level: values["overall"] for level, values in metrics.items()})
